In [ ]:
def parse(lit):
    neg = lit[0] == '~'
    pred, args = lit[neg:].split('(')
    return neg, pred, args[:-1].split(',')

def resolve(c1, c2):
    for l1 in c1:
        n1, p1, a1 = parse(l1)
        for l2 in c2:
            n2, p2, a2 = parse(l2)
            if p1 == p2 and n1 != n2:
                sub = {x: y for x, y in zip(a1, a2) if x.islower() and x != y}
                sub.update({y: x for x, y in zip(a1, a2) if y.islower() and x != y})
                if any(x != y and not x.islower() and not y.islower() for x, y in zip(a1, a2)): continue
                rem = [l for l in c1 if l != l1] + [l for l in c2 if l != l2]
                res = []
                for l in rem:
                    n, p, args = parse(l)
                    res.append(f"{'~' if n else ''}{p}({','.join(sub.get(a, a) for a in args)})")
                return list(set(res))

def prove(kb, goal):
    q, step = [goal], 1
    while q:
        curr = q.pop(0)
        for clause in kb:
            res = resolve(curr, clause)
            if res is not None:
                print(f"{step}. {' | '.join(res) if res else '[] -> PROVED'}")
                step += 1
                if not res: return
                q.append(res)

KB = [
    ["~Food(x)", "Likes(John,x)"],
    ["Food(Apple)"],
    ["Food(Vegetables)"],
    ["~Eats(x,y)", "Killed(x)", "Food(y)"],
    ["Eats(Anil,Peanuts)"],
    ["Alive(Anil)"],
    ["~Eats(Anil,x)", "Eats(Harry,x)"],
    ["~Alive(x)", "~Killed(x)"],
    ["Killed(x)", "Alive(x)"]
]

prove(KB, ["~Likes(John,Peanuts)"])

1. ~Food(Peanuts)
2. Killed(x) | ~Eats(x,Peanuts)
3. Killed(Anil)
4. Killed(Peanuts) | ~Eats(Anil,Peanuts)
5. ~Alive(x) | ~Eats(x,Peanuts)
6. ~Alive(Anil)
7. Killed(Peanuts)
8. ~Alive(Peanuts) | ~Eats(Anil,Peanuts)
9. ~Alive(Anil)
10. ~Eats(Anil,Peanuts)
11. ~Alive(Peanuts) | ~Eats(Anil,Peanuts)
12. Killed(x) | ~Eats(x,Peanuts)
13. [] -> PROVED
